In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('Customer-Churn-Prediction.csv')
#print(df.to_string())

In [ ]:
#print(df.head(10))

In [ ]:
#print(df.info())

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)

In [ ]:
print(df.duplicated().sum())

In [ ]:
#Number of Distinct values in each columns
print(df.nunique())

In [ ]:
#MultipleLines Column contains "NO Phone Service", mapping it to "No"
df['MultipleLines'] = df['MultipleLines'].map({"Yes":"Yes", "No":"No", "No phone service":"No"})

In [ ]:
#Now MultipleLines contains 2 distinct values only
print(df.nunique())

In [ ]:
print(df['MultipleLines'].head())

In [ ]:
#Printing min and max values from tenure
print(df['tenure'].min())
print(df['tenure'].max())

In [ ]:
#Creating a new column "tenure_group" and inserting it.
tenure_group=[]
for i in df['tenure']:
    if(i<=12):
        tenure_group.append('0-12')
    elif(i<=24):
        tenure_group.append('13-24')
    elif(i<=48):
        tenure_group.append('25-48')
    elif(i<=60):
        tenure_group.append('49-60')
    else:
        tenure_group.append('>60')

In [ ]:
import numpy as np
tenure_group=np.array(tenure_group)

In [ ]:
df['tenure_group']=tenure_group

In [ ]:
print(df.nunique())

In [ ]:
#SeniorCitizen has '0' or '1', mapping to 'Yes' or 'No'
df['SeniorCitizen']=df['SeniorCitizen'].map({0:"No", 1:"Yes"})

In [ ]:
print(df.head())

In [ ]:
#Dropping CustomerId and tenure
df.drop(['customerID'], inplace=True, axis=1)
#df.drop(['tenure'], inplace=True, axis=1)
print(df.head())

In [ ]:
#Churn from 'yes' or 'no' to '1' or '0'
df['Churn']=df['Churn'].map({'Yes':1, 'No':0})
print(df['Churn'].head())

In [ ]:
#Correlation Matrix between two numeric variables i.e. monthly charges and total charges
mo=df['MonthlyCharges']
to=df['TotalCharges']
# corr=monthlyCharges.corr(totalCharges)
# print(corr)
print(mo.head())
print(to.head())

In [ ]:
#Since both are of different data types we first typecast totalCharges to float64
print(df.dtypes)

In [ ]:
#Checking whehter TotalCharges containes any value that cannot be typecasted into float
df['TotalCharges']=pd.to_numeric(df['TotalCharges'], errors='coerce') 
#Coerce simply convert non changeable values to NaN to avoid errors

In [ ]:
#There are 11 values which cannot be converted to Float
print(df['TotalCharges'].isna().sum())

In [ ]:
#We drop those rows
df=df.drop(index=df[df['TotalCharges'].isna()].index)
print(df['TotalCharges'].isna().sum())

In [ ]:
print(df.dtypes)

In [ ]:
#Calculating Correlation
mo=df['MonthlyCharges']
to=df['TotalCharges']
corr=to.corr(mo)
print(corr)
#Correlation coeff == 1 implies strong positive correlation, -1 implies strong negative correlation, 0 implies no correlation
#Since it came out to be 0.65 which implies weakly correlated we are gonna keep both variables

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

In [ ]:
#Data Visualization
import matplotlib.pyplot as plt

In [ ]:
plt.hist(df['gender'])
plt.xlabel('Gender')
plt.ylabel('Frequency')
plt.title('Gender Frequency')

In [ ]:
plt.hist(df['SeniorCitizen'])
plt.xlabel('Catagories')
plt.ylabel('Frequency')
plt.title('Age Frequency')

In [ ]:
plt.hist(df['Contract'])
plt.xlabel('tenure')
plt.ylabel('Frequency')
plt.title('Contract Frequency')

In [ ]:
plt.hist(df['InternetService'])
plt.xlabel('Service')
plt.ylabel('Frequency')
plt.title('Internet Service Frequency')

In [ ]:
plt.hist(df['OnlineSecurity'])
plt.xlabel('Categories')
plt.ylabel('Frequency')
plt.title('Security Frequency')

In [ ]:
plt.hist(df['PaymentMethod'])
plt.xlabel('Categories')
plt.ylabel('Frequency')
plt.title('Method Frequency')

In [ ]:
plt.hist(df['tenure_group'])
plt.xlabel('Categories in months')
plt.ylabel('Frequency')
plt.title('Group Frequency')

In [ ]:
plt.hist(df['Churn'])
plt.xlabel('Categories')
plt.ylabel('Frequency')
plt.title('Churn Frequency')

In [ ]:
#Splitting the data in Training and Testing
from sklearn.model_selection import train_test_split as tts
x=df.drop('Churn', axis=1)     #Only Features
y=df['Churn']                  #Target Variable

In [ ]:
#Standardizing the numerical Inputs to Improve the Accuracy of the Model
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x['TotalCharges'] = scaler.fit_transform(x['TotalCharges'])

In [ ]:
#Categorical Data must be coverted into dummy variables before getting feeded into the model
x=pd.get_dummies(x, drop_first=True)

In [ ]:
x_train, x_test, y_train, y_test=tts(x, y, test_size=0.2, random_state=90)

In [ ]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression
#Creating Model
logit=LogisticRegression()
#Fitting the training data to the Model
logit.fit(x_train, y_train)
logit_acc=logit.score(x_test, y_test)
print("Accuracy of Logistic Regression Model = ", logit_acc)

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix
y_pred = logit.predict(x_test)
conf_mat = confusion_matrix(y_test, y_pred)
print(conf_mat)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=85)
dt.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score
y_pred = dt.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

In [ ]:
#Confusion Matrix for Decision Tree
from sklearn.metrics import confusion_matrix
conf_mat_dec = confusion_matrix(y_test, y_pred)
print(conf_mat_dec)

In [ ]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
#Random Forest Classifier with 100 trees.
rf = RandomForestClassifier(n_estimators=50, random_state=90)

In [ ]:
rf.fit(x_train, y_train)

In [ ]:
y_pred_rf=rf.predict(x_test)

In [ ]:
accuracy_rf=rf.score(x_test, y_test)
print(accuracy_rf)

In [ ]:
#Confusion Matrix for Random Forest Classifier
conf_mat_rf = confusion_matrix(y_test, y_pred_rf)
print(conf_mat_rf)